Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Install independencies

In [4]:
!pip install openai --break-system-packages

In [5]:
from openai import OpenAI

client = userdata.get('GOOGLE_API_KEY')


# 简单测试
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello."}],
    max_tokens=10
)
print(response.choices[0].message.content)

Hello! How can I assist you today?


Load the model

Loading DeepSeek-VL2-tiny...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] You are using a model of type `deepseek_vl_v2` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

ValueError: The checkpoint you are trying to load has model type `deepseek_vl_v2` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

For one image

In [6]:
import base64
import re
from openai import OpenAI

client = userdata.get('GOOGLE_API_KEY')

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/31.jpg"
image_data = encode_image(image_path)

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people → output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender or bianxing people → output Transphobia.
Step 4: If neither of the above applies → output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: {Homophobia, Transphobia, Non_LGBT}
Thought: [Give your reason]"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                {"type": "text", "text": prompt_text}
            ]
        }
    ],
    max_tokens=200
)

raw = response.choices[0].message.content.strip()
print("RAW OUTPUT:", raw)

m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
label = m.group(1) if m else None
print("PARSED LABEL:", label)

RAW OUTPUT: Class labels: Non_LGBT  
Thought: The meme does not contain any references or negative connotations towards gay, lesbian, or transgender individuals. The content appears to focus on a political figure in a humorous context without targeting any specific sexual orientation or gender identity.
PARSED LABEL: Non_LGBT


For multi images

In [13]:
import os

# 删掉已有的错误结果
error_json = "/content/drive/MyDrive/GPT4omini_HM_ZeroShot_pred.json"
if os.path.exists(error_json):
    os.remove(error_json)
    print("已删除错误结果")

In [18]:
# 检查 prompt 里有没有非 ASCII 字符
for i, c in enumerate(prompt_text):
    if ord(c) > 127:
        print(f"位置 {i}: 字符 '{c}' 编码 {ord(c)}")

In [19]:
import base64

# 单独测试编码
image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/1.jpg"
with open(image_path, "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

print("image_data 前20字符:", image_data[:20])
print("mime type: image/jpeg")

# 直接构造 url 看看
url = f"data:image/jpeg;base64,{image_data[:10]}"
print("url 前30字符:", url[:30])
print("url 可以 encode ascii?", url.encode("ascii"))

image_data 前20字符: /9j/4AAQSkZJRgABAQAA
mime type: image/jpeg
url 前30字符: data:image/jpeg;base64,/9j/4AA
url 可以 encode ascii? b'data:image/jpeg;base64,/9j/4AAQSk'


In [20]:
import os
import json
import base64
import re
import time
from openai import OpenAI
from tqdm import tqdm

os.environ["PYTHONIOENCODING"] = "utf-8"

client = userdata.get('GOOGLE_API_KEY')

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/GPT4omini_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def call_with_retry(image_data, mime, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_data}"}},
                            {"type": "text", "text": prompt}
                        ]
                    }
                ],
                max_tokens=200
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)  # 第1次等60秒，第2次等120秒
                print(f"  Rate limit，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# ===============================
# 断点续跑：只跳过成功的结果
# ===============================
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    # 只保留成功的，ERROR 的重跑
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(image_dir, img_name)

    try:
        image_data = encode_image(img_path)
        ext = img_name.lower().split(".")[-1]
        mime = "image/png" if ext == "png" else "image/jpeg"

        raw = call_with_retry(image_data, mime, prompt_text)

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        label = m.group(1) if m else "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(3)  # 每张间隔3秒，避免触发限速

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(5)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 0 条，从断点继续...
剩余待处理: 224 张


推理进度:   0%|          | 0/224 [00:00<?, ?it/s]

✅ 1.jpg -> Non_LGBT


推理进度:   0%|          | 1/224 [00:05<21:18,  5.73s/it]

✅ 2.jpg -> Non_LGBT


...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 223/224 [24:30<00:05,  5.12s/it]

✅ 239.jpg -> Non_LGBT


推理进度: 100%|██████████| 224/224 [24:35<00:00,  6.59s/it]


完成！共 224 条结果已保存
  Homophobia: 35
  Non_LGBT: 184
  Transphobia: 4
  UNKNOWN: 1


Check saved or not